# 01_data_prep.ipynb
- Generate mock data 
- Basic cleaning
- Save cleaned data as a CSV or in-memory object

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta, datetime
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)


Matplotlib is building the font cache; this may take a moment.


# Generate synthetic users

In [3]:
n_users = 100
user_ids = [f"user_{i:04d}" for i in range(n_users)]
print(user_ids)

['user_0000', 'user_0001', 'user_0002', 'user_0003', 'user_0004', 'user_0005', 'user_0006', 'user_0007', 'user_0008', 'user_0009', 'user_0010', 'user_0011', 'user_0012', 'user_0013', 'user_0014', 'user_0015', 'user_0016', 'user_0017', 'user_0018', 'user_0019', 'user_0020', 'user_0021', 'user_0022', 'user_0023', 'user_0024', 'user_0025', 'user_0026', 'user_0027', 'user_0028', 'user_0029', 'user_0030', 'user_0031', 'user_0032', 'user_0033', 'user_0034', 'user_0035', 'user_0036', 'user_0037', 'user_0038', 'user_0039', 'user_0040', 'user_0041', 'user_0042', 'user_0043', 'user_0044', 'user_0045', 'user_0046', 'user_0047', 'user_0048', 'user_0049', 'user_0050', 'user_0051', 'user_0052', 'user_0053', 'user_0054', 'user_0055', 'user_0056', 'user_0057', 'user_0058', 'user_0059', 'user_0060', 'user_0061', 'user_0062', 'user_0063', 'user_0064', 'user_0065', 'user_0066', 'user_0067', 'user_0068', 'user_0069', 'user_0070', 'user_0071', 'user_0072', 'user_0073', 'user_0074', 'user_0075', 'user_0076'

# Generate synthetic bills per user

In [6]:
bills_per_user = np.random.poisson(lam=4, size=n_users)  # Avg 4 bills per user
data = []

for user_id, num_bills in zip(user_ids, bills_per_user):
    for _ in range(num_bills):
        bill_amount = round(np.random.uniform(50, 500), 2)
        due_date = datetime(2024, 1, 1) + timedelta(days=int(np.random.uniform(0, 120)))
        delay = np.random.choice([-5, 0, 1, 3, 7, 14], p=[0.1, 0.4, 0.2, 0.15, 0.1, 0.05])
        payment_date = due_date + timedelta(days=int(delay))  
        was_late = int(delay > 0)
        data.append([user_id, bill_amount, due_date, payment_date, was_late])

In [7]:
# Create DataFrame
df = pd.DataFrame(data, columns=["user_id", "bill_amount", "bill_due_date", "payment_date", "was_late"])

# Calculate derived feature: days_late
df["days_late"] = (df["payment_date"] - df["bill_due_date"]).dt.days

# Preview
df.head()

,user_id,bill_amount,bill_due_date,payment_date,was_late,days_late
0,user_0000,471.57,2024-03-28,2024-03-28,0,0
1,user_0000,64.03,2024-02-01,2024-02-02,1,1
2,user_0000,73.14,2024-02-29,2024-03-01,1,1
3,user_0000,200.41,2024-04-02,2024-04-02,0,0
4,user_0001,83.81,2024-03-28,2024-03-28,0,0


# Check data types and missing values

In [8]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 370 entries, 0 to 369
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   user_id        370 non-null    object        
 1   bill_amount    370 non-null    float64       
 2   bill_due_date  370 non-null    datetime64[ns]
 3   payment_date   370 non-null    datetime64[ns]
 4   was_late       370 non-null    int64         
 5   days_late      370 non-null    int64         
dtypes: datetime64[ns](2), float64(1), int64(2), object(1)
memory usage: 17.5+ KB


user_id          0
bill_amount      0
bill_due_date    0
payment_date     0
was_late         0
days_late        0
dtype: int64

# Quick EDA
Distribution of bill amounts

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["bill_amount"], kde=True, bins=30)
plt.title("Distribution of Bill Amounts")
plt.xlabel("Bill Amount ($)")
plt.ylabel("Frequency")
plt.show()

# Late payment rate
late_rate = df["was_late"].mean()
print(f"Late payment rate: {late_rate:.2%}")

# Bills per user
bills_per_user = df["user_id"].value_counts()
plt.figure(figsize=(8, 4))
sns.histplot(bills_per_user, bins=20, kde=False)
plt.title("Number of Bills per User")
plt.xlabel("Bills")
plt.ylabel("Users")
plt.show()
